In [1]:
# ============================================================
# PS S6E5 - exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min
#
# Base:
#   018: exp_20260510_018_realmlp_shared006_pipeline_a_min
#
# Change from 018:
#   - Keep PitStop
#   - Keep PitStop_cat_
#   - Remove _PitStop_cat_count only
#
# Keep:
#   - Race_Compound_
#   - _Race_Compound_TE
#   - Race_Year_
#   - _Race_Year_TE
#   - original split concat policy
#   - RealMLP params
#
# Purpose:
#   Create another RealMLP derivative by weakening only PitStop count encoding.
#   This is not intended to replace 018. It is evaluated as an additive blend material.
# ============================================================

In [2]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 12.9 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.max_columns", 300)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch

try:
    from pytabkit import RealMLP_TD_Classifier
except Exception as e:
    raise RuntimeError(
        "pytabkit is not installed. In Kaggle, run: !pip install -q pytabkit"
    ) from e

In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    USE_ORIGINAL = True
    ORIGINAL_SPLIT = True

    # 023差分:
    # PitStop_cat_ itself remains, but its count encoding is removed.
    DROP_FEATURES_023 = [
        "_PitStop_cat_count",
    ]

    REALMLP_PARAMS = {
        "random_state": SEED,
        "verbosity": 2,
        "val_metric_name": "1-auc_ovr",
        "n_ens": 8,
        "n_epochs": 4,
        "batch_size": 256,
        "use_early_stopping": False,
        "lr": 0.03,
        "wd": 0.018,
        "sq_mom": 0.98,
        "lr_sched": "lin_cos_log_15",
        "first_layer_lr_factor": 0.25,
        "embedding_size": 6,
        "max_one_hot_cat_size": 18,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "p_drop": 0.05,
        "p_drop_sched": "expm4t",
        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "gelu",
        "plr_lr_factor": 0.1151,
        "plr_sigma": 2.33,
        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",
        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],
    }

    TE_CV = 5
    TE_SMOOTH = "auto"

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


def safe_logloss(y_true, pred):
    return log_loss(y_true, clip_pred(pred))


seed_everything(CFG.SEED)

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device count:", torch.cuda.device_count())

torch version: 2.10.0+cu128
cuda available: True
cuda device count: 2


In [6]:
# ============================================================
# Load Data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train_raw = pd.read_csv(COMP_PATH / "train.csv")
test_raw = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train_raw)
if target_col_train != CFG.TARGET:
    train_raw = train_raw.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)
orig_raw = None

if CFG.USE_ORIGINAL and original_path is not None:
    orig_raw = pd.read_csv(original_path)

    if CFG.TARGET not in orig_raw.columns:
        target_col_orig = resolve_target_column(orig_raw)
        orig_raw = orig_raw.rename(columns={target_col_orig: CFG.TARGET})

    if CFG.DANGER_COL in orig_raw.columns:
        orig_raw = orig_raw.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

print("COMP_PATH:", COMP_PATH)
print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("sample_submission:", sample_submission.shape)

if orig_raw is not None:
    print("orig_raw:", orig_raw.shape)
    print("original path:", original_path)

print("target mean train:", train_raw[CFG.TARGET].mean())
if orig_raw is not None:
    print("target mean orig :", orig_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in sample_submission.columns
assert test_raw[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if orig_raw is not None:
    assert CFG.TARGET in orig_raw.columns
    assert CFG.DANGER_COL not in orig_raw.columns

train_ids = train_raw[CFG.ID_COL].copy()
test_ids = test_raw[CFG.ID_COL].copy()


Load Data
Dropped danger column from original: Normalized_TyreLife
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train_raw: (439140, 16)
test_raw : (188165, 15)
sample_submission: (188165, 2)
orig_raw: (101371, 15)
original path: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
target mean train: 0.19898210137996994
target mean orig : 0.25479673673930414


In [7]:
# ============================================================
# Pipeline A Feature Engineering
# ============================================================

def pipeline_a_fe(df: pd.DataFrame, fit: bool = False, state: dict | None = None):
    """
    shared_006 Pipeline A FE.

    Features:
      - floor numeric columns into categorical strings
      - count encoding for base categorical cols + Year_cat_ / PitStop_cat_
      - RaceProgress 200 quantile bins
      - Race x Compound combo
      - Race x Year combo

    Notes:
      - This function mutates a copy in caller side.
      - Fitted state is built from competition train only, following shared_006.
    """
    if state is None:
        state = {}

    df = df.copy()

    cat_cols_base = df.select_dtypes(include=["object"]).columns.tolist()
    num_cols_base = df.select_dtypes(exclude=["object"]).columns.tolist()
    num_cols_base = [c for c in num_cols_base if c not in [CFG.ID_COL, CFG.TARGET]]

    df["_LP_RP"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")

    df["_TL_LP"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")

    for col in num_cols_base + ["_LP_RP", "_TL_LP"]:
        cat_name = f"{col}_cat_" if col in num_cols_base else f"{col[1:]}_cat_"

        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            state[col] = uniques
        else:
            uniques = state[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")

        df[cat_name] = codes.astype(str)

    for col in cat_cols_base + ["Year_cat_", "PitStop_cat_"]:
        cnt_name = f"_{col}_count" if col in cat_cols_base else f"_{col[:-1]}_count"

        if fit:
            cm = df[col].value_counts()
            state[cnt_name] = cm
        else:
            cm = state[cnt_name]

        df[cnt_name] = df[col].map(cm).fillna(0).astype("int32")

    # RaceProgress quantile bin
    col = "RaceProgress"
    nb = 200
    bn = f"{col}_{nb}_qbin_"

    if fit:
        kb = KBinsDiscretizer(
            n_bins=nb,
            encode="ordinal",
            strategy="quantile",
            subsample=None,
        )
        df[bn] = kb.fit_transform(df[[col]]).ravel().astype("int32").astype(str)
        state[bn] = kb
    else:
        df[bn] = state[bn].transform(df[[col]]).ravel().astype("int32").astype(str)

    combos = [
        ("Race", "Compound"),
        ("Race", "Year"),
    ]

    combo_names = []

    for cols in combos:
        cn = "_".join(cols) + "_"
        combo_names.append(cn)

        s = df[cols[0]].astype(str)
        for c in cols[1:]:
            s = s + "_" + df[c].astype(str)

        if fit:
            codes, uniques = pd.factorize(s, sort=False)
            state[cn] = uniques
        else:
            uniques = state[cn]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = s.map(code_map).fillna(-1).astype("int32")

        df[cn] = codes.astype(str)

    return df, state, combo_names

In [8]:
# ============================================================
# Build Pipeline A Features
# ============================================================

print_section("Build Pipeline A Features")

X_a = train_raw.drop([CFG.ID_COL, CFG.TARGET], axis=1).copy()
y_a = train_raw[CFG.TARGET].astype(int).copy()

Xt_a = test_raw.drop([CFG.ID_COL], axis=1).copy()

if orig_raw is not None and CFG.USE_ORIGINAL:
    orig_a = orig_raw.drop([CFG.TARGET], axis=1).copy()
    y_orig_a = orig_raw[CFG.TARGET].astype(int).copy()
else:
    orig_a = None
    y_orig_a = None

X_a, st_a, combo_names = pipeline_a_fe(X_a, fit=True)
Xt_a, _, _ = pipeline_a_fe(Xt_a, fit=False, state=st_a)

if orig_a is not None:
    orig_a, _, _ = pipeline_a_fe(orig_a, fit=False, state=st_a)

# ============================================================
# 023差分: remove _PitStop_cat_count only
# ============================================================

removed_features_023 = []

for c in CFG.DROP_FEATURES_023:
    existed = (
        c in X_a.columns
        or c in Xt_a.columns
        or (orig_a is not None and c in orig_a.columns)
    )

    if c in X_a.columns:
        X_a = X_a.drop(columns=[c])

    if c in Xt_a.columns:
        Xt_a = Xt_a.drop(columns=[c])

    if orig_a is not None and c in orig_a.columns:
        orig_a = orig_a.drop(columns=[c])

    if existed:
        removed_features_023.append(c)

print("023 removed features:", removed_features_023)

# Guardrails
assert "PitStop" in X_a.columns, "023 violation: raw PitStop should remain."
assert "PitStop_cat_" in X_a.columns, "023 violation: PitStop_cat_ should remain."
assert "_PitStop_cat_count" not in X_a.columns
assert "_PitStop_cat_count" not in Xt_a.columns
if orig_a is not None:
    assert "_PitStop_cat_count" not in orig_a.columns

print("Pipeline A train features:", X_a.shape)
print("Pipeline A test features :", Xt_a.shape)
if orig_a is not None:
    print("Pipeline A orig features :", orig_a.shape)

print("combo_names:", combo_names)
print("columns:")
print(X_a.columns.tolist())

print("023 guardrail check:")
print("  PitStop in X_a:", "PitStop" in X_a.columns)
print("  PitStop_cat_ in X_a:", "PitStop_cat_" in X_a.columns)
print("  _PitStop_cat_count in X_a:", "_PitStop_cat_count" in X_a.columns)
print("  Race_Year_ in X_a:", "Race_Year_" in X_a.columns)
print("  Race_Compound_ in X_a:", "Race_Compound_" in X_a.columns)

assert X_a.shape[1] == Xt_a.shape[1]
if orig_a is not None:
    assert X_a.shape[1] == orig_a.shape[1]


Build Pipeline A Features
023 removed features: ['_PitStop_cat_count']
Pipeline A train features: (439140, 36)
Pipeline A test features : (188165, 36)
Pipeline A orig features : (101371, 36)
combo_names: ['Race_Compound_', 'Race_Year_']
columns:
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LP_RP', '_TL_LP', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LP_RP_cat_', 'TL_LP_cat_', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', 'RaceProgress_200_qbin_', 'Race_Compound_', 'Race_Year_']
023 guardrail check:
  PitStop in X_a: True
  PitStop_cat_ in X_a: True
  _PitStop_cat_count in X_a: False
  Race_Year_ in X_a: True
  Race_Compound_ in X_a: True


In [9]:
# ============================================================
# 5fold RealMLP Training
# ============================================================

print_section("5fold RealMLP Training")

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

if orig_a is not None and CFG.USE_ORIGINAL and CFG.ORIGINAL_SPLIT:
    orig_splits = list(
        skf.split(orig_a, y_orig_a)
    )
else:
    orig_splits = None

oof_mlp = np.zeros(len(X_a), dtype=np.float32)
pred_mlp = np.zeros(len(Xt_a), dtype=np.float32)

fold_records = []
te_feature_names_final = []
final_feature_columns = None

start_time = time.time()

for fold, (tri, vi) in enumerate(skf.split(X_a, y_a), 1):
    print("\n" + "=" * 90)
    print(f"RealMLP Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    Xtr_base = X_a.iloc[tri].copy().reset_index(drop=True)
    ytr_base = y_a.iloc[tri].copy().reset_index(drop=True)

    Xvl = X_a.iloc[vi].copy().reset_index(drop=True)
    yvl = y_a.iloc[vi].copy().reset_index(drop=True)

    Xts = Xt_a.copy().reset_index(drop=True)

    if orig_a is not None and CFG.USE_ORIGINAL:
        if CFG.ORIGINAL_SPLIT:
            oti, ovi = orig_splits[fold - 1]
            Xorig_fold = orig_a.iloc[oti].copy().reset_index(drop=True)
            yorig_fold = y_orig_a.iloc[oti].copy().reset_index(drop=True)
            n_orig_used = len(Xorig_fold)
            n_orig_holdout = len(ovi)
        else:
            Xorig_fold = orig_a.copy().reset_index(drop=True)
            yorig_fold = y_orig_a.copy().reset_index(drop=True)
            n_orig_used = len(Xorig_fold)
            n_orig_holdout = 0

        Xtr = pd.concat(
            [Xtr_base, Xorig_fold],
            axis=0,
            ignore_index=True,
        )
        ytr = pd.concat(
            [ytr_base, yorig_fold],
            axis=0,
            ignore_index=True,
        )
    else:
        Xtr = Xtr_base.copy()
        ytr = ytr_base.copy()
        n_orig_used = 0
        n_orig_holdout = 0

    print("Xtr:", Xtr.shape, "Xvl:", Xvl.shape, "Xts:", Xts.shape)
    print("ytr mean:", float(ytr.mean()), "yvl mean:", float(yvl.mean()))
    print("original used:", n_orig_used, "original holdout:", n_orig_holdout)

    # fold-safe TargetEncoder for combo_names
    te = TargetEncoder(
        cv=CFG.TE_CV,
        smooth=CFG.TE_SMOOTH,
        shuffle=True,
        random_state=CFG.SEED,
    )

    te_names = [f"_{c}TE" for c in combo_names]

    Xtr[te_names] = te.fit_transform(Xtr[combo_names], ytr)
    Xvl[te_names] = te.transform(Xvl[combo_names])
    Xts[te_names] = te.transform(Xts[combo_names])

    if not te_feature_names_final:
        te_feature_names_final = te_names

    if final_feature_columns is None:
        final_feature_columns = list(Xtr.columns)

    print("After TE Xtr:", Xtr.shape)
    print("TE features:", te_names)

    seed_everything(CFG.SEED + fold)

    mdl = RealMLP_TD_Classifier(**CFG.REALMLP_PARAMS)

    mdl.fit(Xtr, ytr, Xvl, yvl)

    va_pred = clip_pred(mdl.predict_proba(Xvl)[:, 1]).astype(np.float32)
    te_pred = clip_pred(mdl.predict_proba(Xts)[:, 1]).astype(np.float32)

    oof_mlp[vi] = va_pred
    pred_mlp += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(yvl, va_pred)
    fold_logloss = safe_logloss(yvl, va_pred)

    print(f"Fold {fold} AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.9f}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "n_train_comp": int(len(Xtr_base)),
        "n_train_orig": int(n_orig_used),
        "n_orig_holdout": int(n_orig_holdout),
        "n_valid": int(len(Xvl)),
        "n_features_before_te": int(X_a.shape[1]),
        "n_features_after_te": int(Xtr.shape[1]),
        "target_mean_train": float(ytr.mean()),
        "target_mean_valid": float(yvl.mean()),
    })

    del mdl, Xtr_base, ytr_base, Xvl, yvl, Xts, Xtr, ytr
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

elapsed_sec = time.time() - start_time


5fold RealMLP Training

RealMLP Fold 1/5
Xtr: (432408, 36) Xvl: (87828, 36) Xts: (188165, 36)
ytr mean: 0.2094480213132042 yvl mean: 0.19899121009245344
original used: 81096 original holdout: 20275
After TE Xtr: (432408, 38)
TE features: ['_Race_Compound_TE', '_Race_Year_TE']
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LP_RP', '_TL_LP', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LP_RP_cat_', 'TL_LP_cat_', 'RaceProgress_200_qbin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.054271
Epoch 2/4: val 1-auc_ovr = 0.047651
Epoch 3/4: val 1-auc_ovr = 0.046953
Epoch 4/4: val 1-auc_ovr = 0.045422


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC    : 0.954577978
Fold 1 LogLoss: 0.213707261

RealMLP Fold 2/5
Xtr: (432409, 36) Xvl: (87828, 36) Xts: (188165, 36)
ytr mean: 0.20945216218903862 yvl mean: 0.19897982420184906
original used: 81097 original holdout: 20274
After TE Xtr: (432409, 38)
TE features: ['_Race_Compound_TE', '_Race_Year_TE']
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LP_RP', '_TL_LP', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LP_RP_cat_', 'TL_LP_cat_', 'RaceProgress_200_qbin_', 'Race_Compound_', 'Race_Year_']


...

Fold 4 AUC    : 0.952535887
Fold 4 LogLoss: 0.218360812

RealMLP Fold 5/5
Xtr: (432409, 36) Xvl: (87828, 36) Xts: (188165, 36)
ytr mean: 0.20944984956372323 yvl mean: 0.19897982420184906
original used: 81097 original holdout: 20274
After TE Xtr: (432409, 38)
TE features: ['_Race_Compound_TE', '_Race_Year_TE']
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LP_RP', '_TL_LP', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LP_RP_cat_', 'TL_LP_cat_', 'RaceProgress_200_qbin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.054646
Epoch 2/4: val 1-auc_ovr = 0.047928
Epoch 3/4: val 1-auc_ovr = 0.047243
Epoch 4/4: val 1-auc_ovr = 0.045616


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 AUC    : 0.954384115
Fold 5 LogLoss: 0.214140998


In [10]:
# ============================================================
# CV Result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(y_a, oof_mlp)
cv_logloss = safe_logloss(y_a, oof_mlp)

fold_df = pd.DataFrame(fold_records)

print(f"Overall CV AUC    : {cv_auc:.9f}")
print(f"Overall CV LogLoss: {cv_logloss:.9f}")
print("fold auc mean:", fold_df["auc"].mean())
print("fold auc std :", fold_df["auc"].std())
print("elapsed sec:", elapsed_sec)

display(fold_df)


CV Result
Overall CV AUC    : 0.953446261
Overall CV LogLoss: 0.216210651
fold auc mean: 0.9534563438503065
fold auc std : 0.0009880967986510654
elapsed sec: 480.32294487953186


,fold,auc,logloss,n_train_comp,n_train_orig,n_orig_holdout,n_valid,n_features_before_te,n_features_after_te,target_mean_train,target_mean_valid
0,1,0.954578,0.213707,351312,81096,20275,87828,36,38,0.209448,0.198991
1,2,0.952503,0.218430,351312,81097,20274,87828,36,38,0.209452,0.198980
2,3,0.953280,0.216414,351312,81097,20274,87828,36,38,0.209450,0.198980
3,4,0.952536,0.218361,351312,81097,20274,87828,36,38,0.209450,0.198980
4,5,0.954384,0.214141,351312,81097,20274,87828,36,38,0.209450,0.198980


In [11]:
# ============================================================
# Save Artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof_mlp)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred_mlp)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids.values,
    CFG.TARGET: y_a.values,
    "pred": oof_mlp,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
sub[CFG.TARGET] = clip_pred(pred_mlp)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)

if final_feature_columns is None:
    final_feature_columns = list(X_a.columns) + te_feature_names_final

    # 023 guardrails inside fold
    assert "PitStop" in Xtr.columns
    assert "PitStop_cat_" in Xtr.columns
    assert "_PitStop_cat_count" not in Xtr.columns
    assert "_PitStop_cat_count" not in Xvl.columns
    assert "_PitStop_cat_count" not in Xts.columns

    assert "Race_Year_" in Xtr.columns
    assert "_Race_Year_TE" in Xtr.columns
    assert "Race_Compound_" in Xtr.columns
    assert "_Race_Compound_TE" in Xtr.columns

feature_df = pd.DataFrame({
    "feature": final_feature_columns,
    "is_target_encoding": [c in te_feature_names_final for c in final_feature_columns],
    "is_combo": [c in combo_names for c in final_feature_columns],
    "is_count": [c.endswith("_count") for c in final_feature_columns],
    "is_cat_floor": [c.endswith("_cat_") for c in final_feature_columns],
    "is_qbin": ["_qbin_" in c for c in final_feature_columns],
    "contains_year": ["Year" in c for c in final_feature_columns],
    "contains_pitstop": ["PitStop" in c for c in final_feature_columns],
    "contains_race": ["Race" in c for c in final_feature_columns],
    "contains_compound": ["Compound" in c for c in final_feature_columns],
    "is_removed_pitstop_cat_count": [c == "_PitStop_cat_count" for c in final_feature_columns],
    "is_kept_pitstop_raw": [c == "PitStop" for c in final_feature_columns],
    "is_kept_pitstop_cat": [c == "PitStop_cat_" for c in final_feature_columns],
    "is_kept_race_year_te": [c == "_Race_Year_TE" for c in final_feature_columns],
    "is_kept_race_compound_te": [c == "_Race_Compound_TE" for c in final_feature_columns],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

# 023 guardrails
assert "PitStop" in final_feature_columns
assert "PitStop_cat_" in final_feature_columns
assert "_PitStop_cat_count" not in final_feature_columns

assert "Race_Year_" in final_feature_columns
assert "_Race_Year_TE" in final_feature_columns
assert "Race_Compound_" in final_feature_columns
assert "_Race_Compound_TE" in final_feature_columns

pd.Series(combo_names, name="combo_feature").to_csv(
    CFG.OUTDIR / "combo_features.csv",
    index=False,
)

pd.Series(te_feature_names_final, name="te_feature").to_csv(
    CFG.OUTDIR / "target_encoding_features.csv",
    index=False,
)

pd.Series(list(X_a.columns), name="feature_before_te").to_csv(
    CFG.OUTDIR / "features_before_te.csv",
    index=False,
)

pd.DataFrame({
    "removed_feature": removed_features_023,
    "reason": [
        "023 ablation: remove only PitStop_cat count encoding; keep PitStop and PitStop_cat_"
        for _ in removed_features_023
    ],
}).to_csv(CFG.OUTDIR / "removed_features_023.csv", index=False)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)
display(oof_csv.head())
display(sub.head())
display(feature_df.head(50))


Save Artifacts
Saved to: /kaggle/working/exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min
Submission: /kaggle/working/exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min/submission_exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min.csv


,id,PitNextLap,pred
0,0,1,0.800470
1,1,0,0.623133
2,2,1,0.588509
3,3,0,0.000880
4,4,0,0.246449


,id,PitNextLap
0,439140,0.005658
1,439141,0.018920
2,439142,0.006519
3,439143,0.247335
4,439144,0.758983


,feature,is_target_encoding,is_combo,is_count,is_cat_floor,is_qbin,contains_year,contains_pitstop,contains_race,contains_compound,is_removed_pitstop_cat_count,is_kept_pitstop_raw,is_kept_pitstop_cat,is_kept_race_year_te,is_kept_race_compound_te
0,Driver,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,Compound,False,False,False,False,False,False,False,False,True,False,False,False,False,False
2,Race,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3,Year,False,False,False,False,False,True,False,False,False,False,False,False,False,False
4,PitStop,False,False,False,False,False,False,True,False,False,False,True,False,False,False
5,LapNumber,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6,Stint,False,False,False,False,False,False,False,False,False,False,False,False,False,False
7,TyreLife,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8,Position,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9,LapTime (s),False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [12]:
# ============================================================
# Save cv_result.json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "source": {
        "shared_code": "shared_006",
        "base_experiment": "exp_20260510_018_realmlp_shared006_pipeline_a_min",
        "used_part": "Pipeline A RealMLP only",
        "change_from_018": [
            "Keep raw PitStop.",
            "Keep PitStop_cat_.",
            "Remove _PitStop_cat_count only.",
            "Keep Race_Year_ and _Race_Year_TE.",
            "Keep Race_Compound_ and _Race_Compound_TE.",
        ],
        "excluded_parts": [
            "Pipeline B CatBoost",
            "Pipeline C XGBoost",
            "shared_006 internal 3-model blend",
        ],
        "note": [
            "023 is an 018 derivative.",
            "Only Pipeline A RealMLP is used.",
            "PitStop itself remains.",
            "PitStop_cat_ remains.",
            "_PitStop_cat_count is intentionally removed.",
            "Race_Year_TE is retained; this is not the same as 022.",
            "Original rows are split by StratifiedKFold and only original fold-train side is appended.",
            "Normalized_TyreLife is explicitly dropped from original.",
        ],
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "original_available": orig_raw is not None,
        "original_shape_after_drop": list(orig_raw.shape) if orig_raw is not None else None,
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL,
        "original_split": CFG.ORIGINAL_SPLIT,
        "competition_target_mean": float(train_raw[CFG.TARGET].mean()),
        "original_target_mean": float(orig_raw[CFG.TARGET].mean()) if orig_raw is not None else None,
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": CFG.N_SPLITS,
        "shuffle": True,
        "random_state": CFG.SEED,
        "target_encoding": {
            "library": "sklearn.preprocessing.TargetEncoder",
            "cv": CFG.TE_CV,
            "smooth": CFG.TE_SMOOTH,
            "shuffle": True,
            "random_state": CFG.SEED,
            "columns": combo_names,
            "output_features": te_feature_names_final,
        },
    },
    "features": {
        "pipeline": "shared_006 Pipeline A",
        "feature_count_before_te": int(X_a.shape[1]),
        "feature_count_after_te": int(len(final_feature_columns)),
        "combo_features": combo_names,
        "target_encoding_features": te_feature_names_final,
        "removed_features_023": removed_features_023,
        "pitstop_raw_kept": "PitStop" in final_feature_columns,
        "pitstop_cat_kept": "PitStop_cat_" in final_feature_columns,
        "pitstop_cat_count_used": "_PitStop_cat_count" in final_feature_columns,
        "race_year_te_used": "_Race_Year_TE" in final_feature_columns,
        "race_compound_te_used": "_Race_Compound_TE" in final_feature_columns,
        "feature_blocks": [
            "raw categorical columns",
            "raw numeric columns",
            "LP_RP and TL_LP domain ratios",
            "floor numeric categorical features",
            "count encoding for categorical features except _PitStop_cat_count",
            "RaceProgress 200 quantile bin",
            "Race_Compound combo",
            "Race_Year combo",
            "fold-safe TargetEncoder for Race_Compound and Race_Year",
        ],
        "risk_features_to_watch": [
            "PitStop",
            "PitStop_cat_",
            "Race_Year_",
            "_Race_Year_TE",
            "LapTime_Delta_cat_",
            "Cumulative_Degradation_cat_",
        ],
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
    },
    "model": {
        "family": "RealMLP",
        "estimator": "pytabkit.RealMLP_TD_Classifier",
        "params": CFG.REALMLP_PARAMS,
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
        "elapsed_sec": float(elapsed_sec),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "combo_features": str(CFG.OUTDIR / "combo_features.csv"),
        "target_encoding_features": str(CFG.OUTDIR / "target_encoding_features.csv"),
        "features_before_te": str(CFG.OUTDIR / "features_before_te.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
        "removed_features_023": str(CFG.OUTDIR / "removed_features_023.csv"),
    },
    "decision_policy": {
        "single_model": [
            "Compare against 018 and 022.",
            "023 is not expected to replace 018.",
            "If CV remains above 0.9530, it is a promising RealMLP derivative.",
            "If CV falls below 0.9525, it is probably too weak.",
        ],
        "blend": [
            "Add 023 to current best stack: 007 + 014 + 015 + 016 + 017 + 018 + 020 + 021 + 022.",
            "Evaluate avg / Ridge / ElasticNet / LogReg / HC / NM.",
            "Do not replace 018 initially.",
            "If 023 receives natural positive weight while 018/022 remain useful, keep.",
            "If 023 is zero-weight, stop this PitStop-count branch.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "experiment": {
    "id": "exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC"
  },
  "source": {
    "shared_code": "shared_006",
    "base_experiment": "exp_20260510_018_realmlp_shared006_pipeline_a_min",
    "used_part": "Pipeline A RealMLP only",
    "change_from_018": [
      "Keep raw PitStop.",
      "Keep PitStop_cat_.",
      "Remove _PitStop_cat_count only.",
      "Keep Race_Year_ and _Race_Year_TE.",
      "Keep Race_Compound_ and _Race_Compound_TE."
    ],
    "excluded_parts": [
      "Pipeline B CatBoost",
      "Pipeline C XGBoost",
      "shared_006 internal 3-model blend"
    ],
    "note": [
      "023 is an 018 derivative.",
      "Only Pipeline A RealMLP is used.",
      "PitStop itself remains.",
      "PitStop_cat_ remains.",
      "_PitStop_cat_count is intentionally removed.",
      "Race_Year_TE is retained; this is not the same as 022.",
      "Original rows are

In [13]:
# ============================================================
# Final Preview
# ============================================================

print_section("Final Preview")

print("CV AUC:", cv_auc)
print("CV LogLoss:", cv_logloss)
print("OOF range:", float(oof_mlp.min()), float(oof_mlp.max()), "mean:", float(oof_mlp.mean()))
print("Pred range:", float(pred_mlp.min()), float(pred_mlp.max()), "mean:", float(pred_mlp.mean()))

display(fold_df)
display(oof_csv.head())
display(sub.head())
display(feature_df)


Final Preview
CV AUC: 0.9534462611732204
CV LogLoss: 0.21621065093320208
OOF range: 1.438199615222402e-05 0.9986774325370789 mean: 0.19936390221118927
Pred range: 2.2256417651078664e-05 0.9973777532577515 mean: 0.19848527014255524


,fold,auc,logloss,n_train_comp,n_train_orig,n_orig_holdout,n_valid,n_features_before_te,n_features_after_te,target_mean_train,target_mean_valid
0,1,0.954578,0.213707,351312,81096,20275,87828,36,38,0.209448,0.198991
1,2,0.952503,0.218430,351312,81097,20274,87828,36,38,0.209452,0.198980
2,3,0.953280,0.216414,351312,81097,20274,87828,36,38,0.209450,0.198980
3,4,0.952536,0.218361,351312,81097,20274,87828,36,38,0.209450,0.198980
4,5,0.954384,0.214141,351312,81097,20274,87828,36,38,0.209450,0.198980


,id,PitNextLap,pred
0,0,1,0.800470
1,1,0,0.623133
2,2,1,0.588509
3,3,0,0.000880
4,4,0,0.246449


,id,PitNextLap
0,439140,0.005658
1,439141,0.018920
2,439142,0.006519
3,439143,0.247335
4,439144,0.758983


,feature,is_target_encoding,is_combo,is_count,is_cat_floor,is_qbin,contains_year,contains_pitstop,contains_race,contains_compound,is_removed_pitstop_cat_count,is_kept_pitstop_raw,is_kept_pitstop_cat,is_kept_race_year_te,is_kept_race_compound_te
0,Driver,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,Compound,False,False,False,False,False,False,False,False,True,False,False,False,False,False
2,Race,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3,Year,False,False,False,False,False,True,False,False,False,False,False,False,False,False
4,PitStop,False,False,False,False,False,False,True,False,False,False,True,False,False,False
5,LapNumber,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6,Stint,False,False,False,False,False,False,False,False,False,False,False,False,False,False
7,TyreLife,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8,Position,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9,LapTime (s),False,False,False,False,False,False,False,False,False,False,False,False,False,False
